# 長江杯 2026 — A100 Colab 自動跑批與團隊隊列 Worker（Ultra 5x 1,000 CU 省額度版）

**GPU 選擇與額度保護說明（Ultra 5x = 1,000 Compute Units / 月）**：
- ⭐ **主力推薦 `A100`（開啟 High-RAM）**：100% 兼容 `ESMFold` 與 `BindCraft (CUDA 12.4 + JAX + PyRosetta + SolubleMPNN)`，每小時僅消耗 **~5.4–7.5 CU**（1,000 CU 可跑 **135～185 小時**）。
- ⚠️ **切勿選擇 `G4 (RTX PRO 6000 Blackwell, sm_120)` 跑 BindCraft**：Blackwell 架構不兼容 BindCraft 的 CUDA 12.4 / JAX 內核。
- 🛡️ **自動保存 + 自動釋放 GPU（防空轉燒額度）**：第 4 格預設開啟 `AUTO_UNASSIGN_GPU = True`，任務完成並確認寫入 Google Drive 後，會自動執行 `drive.flush_and_unmount()` 與 `runtime.unassign()` 立即斷開計費。

In [ ]:
#@title 1. 檢查 GPU 架構與 CU 費率、安裝依賴、掛載 Google Drive 並同步 GitHub
import os, subprocess, pathlib, torch
from google.colab import drive

DRIVE_WORKSPACE = "/content/drive/MyDrive/Changjiang_Cup_2026" #@param {type:"string"}
GITHUB_REPO_URL = "https://github.com/David-Liv/changjiang-protein-2026.git" #@param {type:"string"}
REPO_DIR = "/content/changjiang-protein-2026"

print("=== [1/4] GPU 架構與 1,000 CU 額度自檢 ===")
!nvidia-smi
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 1e9
    sm = f"sm_{p.major}{p.minor}"
    print(f"檢測到 GPU: {p.name} | 顯存: {gb:.1f} GB | 架構: {sm}")
    if p.major >= 12 or "BLACKWELL" in p.name.upper() or "RTX PRO 6000" in p.name.upper():
        print("❌ [警告] 當前為 G4 (Blackwell sm_120)！BindCraft (CUDA 12.4/JAX) 不支持此架構，請切換為 A100！")
    elif "H100" in p.name.upper() or p.major == 9:
        print("⚠️ [提示] 當前為 H100 (~18 CU/h，1,000 CU 約可跑 55 小時)。日常跑批建議改選 A100 以節省額度。")
    else:
        print("✅ [最佳配置] A100 架構完全兼容 ESMFold 與 BindCraft（~5.4–7.5 CU/h，1,000 CU 可跑 135–185 小時）✓")
else:
    print("❌ 未檢測到 GPU！請在『執行階段 -> 變更執行階段類型』選擇 A100 GPU。")

print("\n=== [2/4] 安裝 ESMFold / ipSAE 評估依賴 (transformers & accelerate) ===")
!pip install -q -U transformers accelerate numpy

print("\n=== [3/4] 掛載 Google Drive ===")
drive.mount('/content/drive')
pathlib.Path(DRIVE_WORKSPACE).mkdir(parents=True, exist_ok=True)

print("\n=== [4/4] 同步 GitHub 最新 Pipeline 代碼 ===")
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!python3 scripts/tnf_pipeline.py selftest
!python3 pipeline/worker.py --drive-root "{DRIVE_WORKSPACE}" --once --dry-run

In [ ]:
#@title 2. 安裝 BindCraft + PyRosetta + SolubleMPNN 環境（僅跑生成任務時需要；若只跑 TNF-α 校準可跳過此格）
import os, torch
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 12:
    raise RuntimeError("當前 GPU 為 G4 (Blackwell sm_120)，與 BindCraft CUDA 12.4 不兼容！請先切換執行階段為 A100 再安裝。")
if not os.path.exists("/content/BindCraft"):
    !git clone https://github.com/martinpacesa/BindCraft /content/BindCraft
    !bash /content/BindCraft/install_bindcraft.sh --cuda '12.4' --pkg_manager 'conda'
else:
    print("BindCraft 已安裝 ✓")

In [ ]:
#@title 3. 將預設任務模板投遞到網盤 `jobs/pending/`（可選：靶點 2 TNF-α 校準 / 靶點 5 TL1A / 靶點 1 EGFR）
import shutil, pathlib

TEMPLATE_CHOICE = "target2_TNFa_calibration" #@param ["target2_TNFa_calibration", "target5_TL1A_minibinder", "target1_EGFR_vhh", "none"]
pending_dir = pathlib.Path(DRIVE_WORKSPACE) / "jobs" / "pending"
pending_dir.mkdir(parents=True, exist_ok=True)

mapping = {
    "target2_TNFa_calibration": "/content/changjiang-protein-2026/jobs/templates/job_target2_TNFa_calibration.json",
    "target5_TL1A_minibinder": "/content/changjiang-protein-2026/jobs/templates/job_target5_TL1A_minibinder.json",
    "target1_EGFR_vhh": "/content/changjiang-protein-2026/jobs/templates/job_target1_EGFR_vhh.json",
}
if TEMPLATE_CHOICE in mapping:
    src = pathlib.Path(mapping[TEMPLATE_CHOICE])
    dst = pending_dir / src.name
    shutil.copy2(src, dst)
    print(f"已投遞任務單至隊列：{dst}")

In [ ]:
#@title 4. 啟動隊列執行器（跑完自動保存至 Google Drive + 自動釋放 GPU 節省 1,000 CU 額度）
import os, subprocess, time, pathlib
from google.colab import drive, runtime

AUTO_UNASSIGN_GPU = True #@param {type:"boolean"}

!git -C /content/changjiang-protein-2026 pull --ff-only

cmd = [
    "python3",
    "/content/changjiang-protein-2026/pipeline/worker.py",
    "--drive-root",
    DRIVE_WORKSPACE,
    "--once",
]
if AUTO_UNASSIGN_GPU:
    cmd.append("--unassign-on-exit")

try:
    subprocess.run(cmd, check=False)
finally:
    if hasattr(os, "sync"):
        os.sync()
    runs_dir = pathlib.Path(DRIVE_WORKSPACE) / "runs"
    saved_count = sum(1 for p in runs_dir.rglob("*") if p.is_file()) if runs_dir.exists() else 0
    print(f"\n✅ 任務執行完畢！已確認 {saved_count} 個結果檔案保存在 Google Drive: {runs_dir}")
    if AUTO_UNASSIGN_GPU and saved_count > 0:
        print("🔒 正在強制刷新 Google Drive 緩存並自動釋放 GPU (runtime.unassign())，避免空轉扣費...")
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        time.sleep(2)
        runtime.unassign()